# Analytical PIAB3D eigenfunction slices

**Prerequisite:** [Analytical PIAB3D eigenfunction slices](../../../docs/lecture-notes/qm/piab3d/piab3d__eigenfunctions.md)

This laboratory evaluates Cartesian plane slices of normalized product eigenfunctions for the nondimensional unit cube with homogeneous Dirichlet boundaries.

## Learning objectives

1. Evaluate selected $(n_x,n_y,n_z)$ states on an explicit plane.
2. Verify boundary zeros and the analytical probability weight of a slice.
3. Relate the two in-plane quantum numbers to visible nodal structure.
4. Compare `X`-, `Y`-, and `Z`-normal slices without changing evaluators.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab3d.base import Piab3D
from projectkoios.physkit.qm.piab3d.tise.analytical.eigenfunctions import (
    Piab3DAnalyticalEigenfunction,
    Piab3DPlaneNormal,
)
from projectkoios.physkit.units import (
    ScalarQuantity,
    Unitless,
    UnitSystem,
    VectorQuantity,
)

In [ ]:
unit_box_length = 1.0
unit_particle_mass = 1.0
model = Piab3D(
    length_x=unit_box_length,
    length_y=unit_box_length,
    length_z=unit_box_length,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
sample_count = 201
coordinate_values = np.linspace(
    start=0.0,
    stop=unit_box_length,
    num=sample_count,
    dtype=np.float64,
)
coordinates = VectorQuantity(
    magnitude=coordinate_values,
    unit=Unitless(),
)
midplane_coordinate = unit_box_length / 2.0
z_coordinate = ScalarQuantity(
    magnitude=midplane_coordinate,
    unit=Unitless(),
)
ground_quantum_number = 1
second_quantum_number = 2
selected_states = (
    (ground_quantum_number, ground_quantum_number, ground_quantum_number),
    (ground_quantum_number, second_quantum_number, ground_quantum_number),
    (second_quantum_number, ground_quantum_number, ground_quantum_number),
    (second_quantum_number, second_quantum_number, ground_quantum_number),
)

slice_results = [
    Piab3DAnalyticalEigenfunction(
        model=model, quantum_number_x=n_x, quantum_number_y=n_y, quantum_number_z=n_z
    ).evaluate_plane_slice(
        plane_normal=Piab3DPlaneNormal.Z,
        coordinates_u=coordinates,
        coordinates_v=coordinates,
        fixed_coordinate=z_coordinate,
    )
    for n_x, n_y, n_z in selected_states
]

In [ ]:
represented_slice_weights = []
for result in slice_results:
    density = result.probability_density.magnitude
    integrated_y = np.trapezoid(density, x=coordinate_values, axis=1)
    represented_slice_weights.append(
        float(np.trapezoid(integrated_y, x=coordinate_values))
    )
    assert np.allclose(result.amplitudes.magnitude[[0, -1], :], 0.0)
    assert np.allclose(result.amplitudes.magnitude[:, [0, -1]], 0.0)

expected_slice_density = 2.0 / model.length_z
normalization_tolerance = 3e-15
assert np.allclose(
    represented_slice_weights,
    expected_slice_density,
    rtol=0.0,
    atol=normalization_tolerance,
)
represented_slice_weights

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

for axis, result in zip(axes.ravel(), slice_results, strict=True):
    image = axis.pcolormesh(
        coordinate_values,
        coordinate_values,
        result.probability_density.magnitude.T,
        shading="auto",
        cmap="viridis",
    )
    axis.set(
        xlabel="$x/L_x$",
        ylabel="$y/L_y$",
        title=(
            rf"$(n_x,n_y,n_z)=({result.quantum_number_x},"
            rf"{result.quantum_number_y},{result.quantum_number_z})$"
        ),
        aspect="equal",
    )
    figure.colorbar(image, ax=axis, label=r"$|\psi(x,y,z=L_z/2)|^2$")

figure.tight_layout()
plt.show()

## Comparing Cartesian plane normals

The same evaluator selects any coordinate as the fixed direction. The ordered in-plane coordinates are $(y,z)$ for normal `X`, $(x,z)$ for normal `Y`, and $(x,y)$ for normal `Z`. The fixed positions below avoid a nodal plane for the $(1,2,3)$ state.


In [ ]:
third_quantum_number = 3
quarter_plane_coordinate = unit_box_length / 4.0
normal_selections = (
    (Piab3DPlaneNormal.X, midplane_coordinate),
    (Piab3DPlaneNormal.Y, quarter_plane_coordinate),
    (Piab3DPlaneNormal.Z, midplane_coordinate),
)
normal_slices = [
    Piab3DAnalyticalEigenfunction(
        model=model,
        quantum_number_x=ground_quantum_number,
        quantum_number_y=second_quantum_number,
        quantum_number_z=third_quantum_number,
    ).evaluate_plane_slice(
        plane_normal=normal,
        coordinates_u=coordinates,
        coordinates_v=coordinates,
        fixed_coordinate=ScalarQuantity(
            magnitude=fixed_value,
            unit=Unitless(),
        ),
    )
    for normal, fixed_value in normal_selections
]

assert [result.in_plane_axis_labels for result in normal_slices] == [
    ("y", "z"),
    ("x", "z"),
    ("x", "y"),
]
assert np.allclose(
    [result.expected_integrated_density.magnitude for result in normal_slices],
    expected_slice_density,
    rtol=0.0,
    atol=normalization_tolerance,
)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)

for axis, result in zip(axes, normal_slices, strict=True):
    axis_u, axis_v = result.in_plane_axis_labels
    image = axis.pcolormesh(
        coordinate_values,
        coordinate_values,
        result.probability_density.magnitude.T,
        shading="auto",
        cmap="magma",
    )
    axis.set(
        xlabel=rf"${axis_u}/L_{axis_u}$",
        ylabel=rf"${axis_v}/L_{axis_v}$",
        title=f"normal {result.plane_normal.value.upper()}",
        aspect="equal",
    )
    figure.colorbar(image, ax=axis, label=r"$|\psi|^2$")

figure.tight_layout()
plt.show()

## Interpretation

At $z=L_z/2$, states with $n_z=1$ have their maximum $z$-factor magnitude. Their integrated planar density is therefore $2/L_z$, not one. The $x$ and $y$ quantum numbers independently control the in-plane nodal lines. A state with even $n_z$ would vanish on this selected midplane.

The second figure demonstrates that plane orientation is selected explicitly while the evaluator retains deterministic in-plane coordinates. All heatmaps are physical-coordinate slices through normalized three-dimensional states. They must not be interpreted as separately normalized two-dimensional quantum states.

## Exercises

1. Evaluate an $n_z=2$ state at the midplane and explain the zero slice.
2. Move the plane to $z=L_z/4$ and calculate the expected slice weight.
3. Compare permutation-related cubic states on the same plane.
4. Integrate sampled slices over $z$ to reconstruct the full three-dimensional normalization.
